# Data cleaning & manipulation

**Notebook Overview**

- This notebook prepares the dataset for a regression model that predicts duration_difference_min.
- The workflow is structured into modular functions, each handling a specific step:
    - Data type fixing and standardization
    - Missing values handling
    - Outlier correction and filtering
    - Feature engineering
    - Categorical encoding
- After preprocessing, irrelevant and leakage-prone columns are removed.
- Feature selection is performed using multiple regression-based methods, and results are combined into a summary table.
- Finally, XGBoost models are trained on different feature subsets and compared using RMSE, MAE, and R².

## Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.data_cleaning_and_manipulations import impute_by_agency_line_hour
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import math

## Functions

In [2]:
def fix_data_types(df):
    """Fix all column types and rename columns"""
    
    for col in df.select_dtypes(include=["datetime"]).columns:
        df[col] = df[col].dt.strftime("%Y-%m-%d %H:%M:%S")
    
    # Hour
    if 'hour_rounded' in df.columns:
        df = df.rename(columns={'hour_rounded': 'full_hour'})
    
    
    # Day - categorical with order
    day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    df['day'] = pd.Categorical(df['day'], categories=day_order, ordered=True)
    
    # String columns
    str_cols = ['line_name', 'alternative', 'agency_name', 'origin_city', 
                'origin_station', 'destination_city', 'destination_station', 'route_type']
    df[str_cols] = df[str_cols].astype(str)
    df['route_type'] = df['route_type'].str.strip()

    
    
    return df


def handle_missing_values(df, ref_df=None, printing_missing_values=False):
    """
    Handle missing values.

    ref_df = Train df (for calculating medians/means)
    If ref_df=None, uses df itself (for train only)

    printing_missing_values:
        If True, prints missing values before and after the process.
    """

    if ref_df is None:
        ref_df = df

    if printing_missing_values:
        print("Missing values BEFORE handling:")
        print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))
        print("-" * 50)

    # Convert day to string for groupby
    df['day'] = df['day'].astype(str)

    # Total_Passengers - progressive groupby imputation
    for col in ['Total_Passengers']:
        df[col] = df.groupby(['route_id', 'direction', 'day', 'full_hour'])[col].transform(
            lambda x: x.fillna(x.median())
        )

        df[col] = df.groupby(['route_id', 'direction', 'day'])[col].transform(
            lambda x: x.fillna(x.median())
        )

        df[col] = df.groupby(['route_id'])[col].transform(
            lambda x: x.fillna(x.median())
        )

        df[col] = df[col].fillna(ref_df[col].median())

    # Avg_Passengers_Per_Bus
    df['Avg_Passengers_Per_Bus'] = df.groupby('route_id')['Avg_Passengers_Per_Bus'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Avg_Passengers_Per_Bus'] = df['Avg_Passengers_Per_Bus'].fillna(
        ref_df['Avg_Passengers_Per_Bus'].median()
    )

    # circular_route
    df['circular_route'] = df['circular_route'].fillna(0)

    # Text columns
    df["line_name"] = df["line_name"].fillna("unknown")
    df["agency_name"] = df["agency_name"].fillna("unknown")
    df["line_num"] = df["line_num"].fillna("unknown")

    # Speed and duration-related columns
    for col in ['speed_kmh_actual', 'duration_min_actual', 'duration_difference_min']:
        df[col] = df.groupby(['route_id', 'direction'])[col].transform(
            lambda x: x.fillna(x.median())
        )
        df[col] = df[col].fillna(ref_df[col].median())

    # Geometric columns
    geo_cols = ['curvity', 'route_length', 'length_in_buffer_m']
    for col in geo_cols:
        if col in df.columns:
            df[col] = df[col].fillna(ref_df[col].mean())

    # Fix route_length zeros/NaNs using route_length_km
    if 'route_length' in df.columns and 'route_length_km' in df.columns:
        zero_mask = (df['route_length'] == 0) | (df['route_length'].isna())
        df.loc[zero_mask, 'route_length'] = df.loc[zero_mask, 'route_length_km'] * 1000

    if printing_missing_values:
        print("Missing values AFTER handling:")
        print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))
        print("-" * 50)

    return df

def handle_outliers(df, boxplots=False, boxplot_cols=None, verbose=False):
    """
    Handle outliers:
    - Fix speed_kmh_planned > 100
    - Fix trips ending after midnight
    - Remove rows where duration_difference_min > 120 or < -120
    
    Parameters:
    df : DataFrame
    boxplots : bool
        If True, plots boxplots BEFORE and AFTER outlier handling.
    boxplot_cols : list
        Columns to include in boxplots.
    verbose : bool
        If True, prints process details.
    """

    # Default columns
    if boxplot_cols is None:
        boxplot_cols = [
            'full_hour',
            'route_length_km',
            'number_of_stops',
            'rainfall_mm',
            'Total_Passengers',
            'curvity',
            'duration_min_planned',
            'duration_min_actual',
            'speed_kmh_planned',
            'speed_kmh_actual',
            'duration_difference_min'
        ]

    # Keep only existing columns
    boxplot_cols = [col for col in boxplot_cols if col in df.columns]

    # 🔹 BEFORE plots
    if boxplots:
        print("\nBoxplots BEFORE outlier handling:")
        plot_boxplots_with_outliers(df, boxplot_cols)

    # --- Outlier handling ---

    # Fix speed > 100
    mask_high = df['speed_kmh_planned'] > 100
    if verbose:
        print(f"Rows with speed > 100: {mask_high.sum()}")
    df.loc[mask_high, 'speed_kmh_planned'] = df.loc[mask_high, 'speed_kmh_planned'] / 1000

    # Fix trips ending after midnight
    df['departure_time_planned'] = pd.to_datetime(df['departure_time_planned'].astype(str), format='mixed')
    df['arrival_time_planned'] = pd.to_datetime(df['arrival_time_planned'].astype(str), format='mixed')

    mask_midnight = df['arrival_time_planned'] < df['departure_time_planned']
    if verbose:
        print(f"Trips ending after midnight: {mask_midnight.sum()}")

    df.loc[mask_midnight, 'duration_min_planned'] = (
        (df.loc[mask_midnight, 'arrival_time_planned'] + pd.Timedelta(days=1)) -
        df.loc[mask_midnight, 'departure_time_planned']
    ).dt.total_seconds() / 60

    df.loc[mask_midnight, 'speed_kmh_planned'] = (
        (df.loc[mask_midnight, 'route_length'] / 1000) /
        (df.loc[mask_midnight, 'duration_min_planned'] / 60)
    )

    # Remove duration_difference_min outliers
    before = len(df)

    df = df[
        (df['duration_difference_min'] >= -120) &
        (df['duration_difference_min'] <= 120)
    ].copy()

    after = len(df)

    if verbose:
        print(f"Rows removed: {before - after:,} ({(before - after)/before*100:.2f}%)")
        print(f"Rows remaining: {after:,}")

    # 🔹 AFTER plots
    if boxplots:
        print("\nBoxplots AFTER outlier handling:")
        plot_boxplots_with_outliers(df, boxplot_cols)

    return df

def add_features(df):
    """Add new features"""
    df_copy = df.copy()
    peak_hours = [7, 8, 9, 14, 15, 16, 17]

    # Peak hour flag
    df_copy['is_peak_hour'] = df_copy['full_hour'].isin(peak_hours).astype(int)

    # Urban flag
    df_copy['urban'] = (df_copy['route_length'] <= 25000).astype(int)

    # Perc within PT route - safe division
    df_copy['perc_within_pt_route'] = np.where(
        df_copy['route_length'] > 0,
        df_copy['length_in_buffer_m'] / df_copy['route_length'],
        np.nan
    )

    # Remove possible inf values
    df_copy['perc_within_pt_route'] = df_copy['perc_within_pt_route'].replace(
        [np.inf, -np.inf], np.nan
    )

    # Optional: fill remaining NaN with 0
    df_copy['perc_within_pt_route'] = df_copy['perc_within_pt_route'].fillna(0)

    # Optional: clip unrealistic values
    df_copy['perc_within_pt_route'] = df_copy['perc_within_pt_route'].clip(0, 2)

    # Peak weighted perc - vectorized
    df_copy['perc_within_pt_route_peak'] = (
        df_copy['perc_within_pt_route'] * df_copy['is_peak_hour']
    )

    # Interaction features
    df_copy['passengers_x_peak'] = df_copy['Total_Passengers'] * df_copy['is_peak_hour']
    df_copy['stops_x_passengers'] = df_copy['number_of_stops'] * df_copy['Total_Passengers']

    # Arrival and departure hour features
    df_copy['departure_hour'] = pd.to_datetime(
        df_copy['departure_time_planned'], format='mixed'
    ).dt.hour

    df_copy['arrival_hour'] = pd.to_datetime(
        df_copy['arrival_time_planned'], format='mixed'
    ).dt.hour

    # Combined categorical feature
    df_copy['agency_linenum_dir_alter'] = (
        df_copy['agency_name'].astype(str) +
        df_copy['line_num'].astype(str) +
        df_copy['direction'].astype(str) +
        df_copy['alternative'].astype(str)
    )

    return df_copy


def encode_categorical_columns(df, te=None, alternative_cols=None):
    """
    Encode categorical columns.
    """
    # הסר כפילויות לפני הכל
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Ordinal encoding - day
    day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    day_mapping = {day: i for i, day in enumerate(day_order)}
    df['day_encoded'] = df['day'].map(day_mapping)
    
    
    # Target encoding
    from category_encoders import TargetEncoder
    target_cols = ['agency_name', 'origin_city', 'destination_city', 
                   'origin_station', 'destination_station','agency_linenum_dir_alter']
    
    if te is None:
        te = TargetEncoder()
        df[[f'{col}_encoded' for col in target_cols]] = te.fit_transform(
            df[target_cols], df['duration_difference_min']
        )
    else:
        df[[f'{col}_encoded' for col in target_cols]] = te.transform(
            df[target_cols]
        ) 
    
    # הסר כפילויות בסוף
    df = df.loc[:, ~df.columns.duplicated()]
    
    return df, te, alternative_cols




def plot_boxplots_with_outliers(df, cols, n_cols=2, figsize=(20, 24)):
    """
    Plot boxplots with outlier statistics (IQR method)

    Parameters:
    df : DataFrame
    cols : list of column names
    n_cols : number of subplot columns
    figsize : figure size
    """
    
    n_rows = math.ceil(len(cols) / n_cols)
    
    plt.figure(figsize=figsize)
    
    for i, col in enumerate(cols):
        ax = plt.subplot(n_rows, n_cols, i + 1)
        
        sns.boxplot(data=df, x=col, ax=ax, color='navy')
        
        # IQR calculation
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        n_outliers = len(df[(df[col] < lower) | (df[col] > upper)])
        pct_outliers = (n_outliers / len(df)) * 100
        
        ax.set_title(
            f'{col}\nOutliers: {n_outliers:,} ({pct_outliers:.1f}%)',
            fontsize=11
        )
        ax.set_xlabel(col)
    
    plt.subplots_adjust(hspace=0.5, wspace=0.3)
    plt.tight_layout()
    plt.show()

def drop_unnecessary_columns(df):
    """Drop columns that are already encoded, not relevant, or cause data leakage"""
    
    cols_to_drop = [
        # כבר מקודדות
        'day', 'alternative', 'agency_name', 'origin_city',
        'origin_station', 'destination_city', 'destination_station',
        # זהות לעמודות אחרות
        'route_mkt', 'route_length_kn',
        # טקסט/זמן
        'date', 'line_name', 'departure_time_planned',
        'arrival_time_planned', 'route_type', 'line_num','agency_linenum_dir_alter',
        # Data Leakage
        'duration_min_actual', 'duration_min_planned', 'speed_kmh_actual',

    ]
    
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    return df

def run_feature_selection_methods(X_train, y_train):
    """
    Run multiple feature selection methods for a regression problem.

    Returns:
    selection_df : DataFrame with selected/not selected features per method
    """

    import numpy as np
    import pandas as pd

    from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import Lasso, Ridge
    from sklearn.svm import LinearSVR
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    methods = [
        "RandomForest",
        "GradientBoost",
        "Lasso",
        "SVR",
        "Ridge"
    ]

    print("Feature selection methods to be used:")
    for i, method in enumerate(methods, start=1):
        print(f"{i}. {method}")

    print("-" * 50)

    total_methods = len(methods)
    finished = 0

    # 1. Random Forest
    rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)
    rf_selected = (rf.feature_importances_ > 0).astype(int)

    finished += 1
    print(f"Finished RandomForest ({finished}/{total_methods})")

    # 2. Gradient Boosting
    gb = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
    gb_selected = (gb.feature_importances_ > 0).astype(int)

    finished += 1
    print(f"Finished GradientBoost ({finished}/{total_methods})")

    # 3. Lasso
    lasso = make_pipeline(
        StandardScaler(),
        Lasso(alpha=0.1, max_iter=10000, random_state=42)
    )

    lasso.fit(X_train, y_train)

    lasso_coef = lasso.named_steps["lasso"].coef_
    lasso_selected = (np.abs(lasso_coef) > 0).astype(int)

    finished += 1
    print(f"Finished Lasso ({finished}/{total_methods})")

    # 4. Linear SVR
    svr = make_pipeline(
        StandardScaler(),
        LinearSVR(C=0.01, max_iter=10000, random_state=42)
    )

    svr.fit(X_train, y_train)

    svr_coef = svr.named_steps["linearsvr"].coef_
    svr_selected = (np.abs(svr_coef) > 0).astype(int)

    finished += 1
    print(f"Finished SVR ({finished}/{total_methods})")

    # 5. Ridge
    ridge = make_pipeline(
        StandardScaler(),
        Ridge(alpha=0.01)
    )

    ridge.fit(X_train, y_train)

    ridge_coef = ridge.named_steps["ridge"].coef_
    ridge_selected = (np.abs(ridge_coef) > 0).astype(int)

    finished += 1
    print(f"Finished Ridge ({finished}/{total_methods})")

    # Create results DataFrame
    selection_df = pd.DataFrame({
        "Feature": X_train.columns,
        "Lasso": lasso_selected,
        "SVR": svr_selected,
        "GradientBoost": gb_selected,
        "RandomForest": rf_selected,
        "Ridge": ridge_selected
    })

    # Sum selections
    selection_cols = ["Lasso", "SVR", "GradientBoost", "RandomForest", "Ridge"]

    selection_df["Sum"] = selection_df[selection_cols].sum(axis=1)

    selection_df = selection_df.sort_values(
        by="Sum",
        ascending=False
    ).reset_index(drop=True)

    return selection_df


def manipulate_df_process(df):
    df = fix_data_types(df)
    df = handle_missing_values(df, ref_df=None)
    df = handle_outliers(df)
    df = add_features(df)
    df, te, alternative_cols = encode_categorical_columns(df, te=None, alternative_cols=None)
    
    return df


def compare_xgb_feature_sets(X_train, y_train, X_val, y_val, selection_df):
    """
    Compare XGBoost performance on:
    1. All features
    2. Features selected by all models (Sum == 5)
    3. Features selected by >=4 models

    Returns:
    results_df
    """

    import numpy as np
    import pandas as pd
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    results = []

    # Define feature sets
    feature_sets = {
        "All Features": X_train.columns.tolist(),
        "Sum == 5": selection_df.loc[selection_df["Sum"] == 5, "Feature"].tolist(),
        "Sum >= 4": selection_df.loc[selection_df["Sum"] >= 4, "Feature"].tolist()
    }

    print("Running XGBoost on feature sets:")
    for name, feats in feature_sets.items():
        print(f"- {name}: {len(feats)} features")

    print("-" * 50)

    # Loop over feature sets
    for name, feats in feature_sets.items():

        if len(feats) == 0:
            print(f"Skipping {name} (no features)")
            continue

        # Subset data
        X_tr = X_train[feats]
        X_vl = X_val[feats]

        # Model
        model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42
        )

        model.fit(X_tr, y_train)

        y_pred = model.predict(X_vl)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)

        print(f"{name} → RMSE: {rmse:.3f}, MAE: {mae:.3f}, R²: {r2:.3f}")

        results.append({
            "Feature_Set": name,
            "Num_Features": len(feats),
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

    results_df = pd.DataFrame(results).sort_values(by="RMSE")

    return results_df

## Imports

In [3]:
train_df = pd.read_csv(r'../data/model_datasets/train_df.csv', encoding='utf-8-sig')
print(f"Train:      {len(train_df):,} rows")

Train:      63,616 rows


## 1. Manage Data types

**Step 1: Fix data types and standardize columns**
- Convert datetime columns to string format
- Rename 'hour_rounded' to 'full_hour' for consistency
- Convert 'day' to an ordered categorical variable (Sunday → Saturday)
- Ensure key categorical columns are treated as strings
- Clean text fields (e.g., remove whitespace from 'route_type')

In [4]:
# Run
train_df = fix_data_types(train_df)

## 2. Missing values handling

**Handle Missing Values**

- Impute missing values using hierarchical group-based strategies (from granular to general levels)
- Apply fallback imputation using global statistics from a reference dataset (train set)
- Handle categorical missing values by assigning default labels (e.g., "unknown")
- Fill binary/indicator features with default values where appropriate
- Ensure consistency in numerical features using group-level and global aggregations
- Correct invalid or inconsistent values (e.g., zero or missing lengths)
- Optionally print missing values summary before and after processing

In [5]:
train_df = handle_missing_values(train_df, printing_missing_values=True)

Missing values BEFORE handling:
Total_Passengers           813
Avg_Passengers_Per_Bus     813
length_in_buffer_m         119
curvity                    119
circular_route             119
duration_min_actual         89
duration_difference_min     89
speed_kmh_actual            89
route_length                 4
dtype: int64
--------------------------------------------------
Missing values AFTER handling:
Series([], dtype: int64)
--------------------------------------------------


## 3. Outliers handling

**Handle Outliers**

- Identify and correct unrealistic values in key features
- Adjust planned trips that cross midnight by recalculating duration and speed
- Remove extreme values in the target variable based on defined thresholds
- Report number and proportion of removed observations
- Optionally visualize feature distributions before and after outlier handling using boxplots

In [6]:
train_df = handle_outliers(train_df, boxplots=False)

## 4. Add Features

**Add Features**

- Create peak-hour indicator based on predefined morning and afternoon peak periods
- Create urban/interurban indicator based on route length
- Calculate the share of the route located within the public transport buffer
- Create peak-weighted route-buffer feature
- Add interaction features between passengers, stops, and peak-hour conditions
- Extract planned departure and arrival hour features
- Create combined categorical route identifier using agency, line, direction, and alternative values

In [7]:
train_df = add_features(train_df)

## 5. Categorical Data Handling

**Encode Categorical Columns**

- Remove duplicate columns before and after the encoding process
- Convert day into an ordinal numerical feature based on weekday order
- Apply target encoding to high-cardinality categorical features
- Fit the target encoder on the training dataset only
- Apply the same encoder to validation and test datasets to avoid data leakage
- Return the transformed dataset along with the fitted encoder for reuse

In [8]:
# Run
train_df, te, alternative_cols = encode_categorical_columns(train_df, te=None, alternative_cols=None)

## 6. Feature Selection

**Feature Selection Process** 

-  Drop columns that are not suitable for modeling:
    - Original categorical columns that were already encoded
    - Raw text and datetime columns that cannot be used directly by the model
    - Duplicate or redundant columns
    - Data leakage columns, such as actual duration and actual speed, that directly reveal  information related to the target
- Define the target variable: duration_difference_min
- Create X_train and y_train after removing the target column
- Run several feature selection methods for a regression problem:
    - Random Forest feature importance
    - Gradient Boosting feature importance
    - Lasso coefficient selection
    - Linear SVR coefficient selection
    - Ridge coefficient ranking
-  Store the result of each method in a feature-selection summary table
-  Calculate a final Sum score for each feature, based on how many methods selected it
-  Compare XGBoost model performance using different feature sets:
    - All available features
    - Features selected by all methods (Sum == 5)
    - Features selected by at least four methods (Sum >= 4)
-  Evaluate and compare the results using RMSE, MAE, and R².

In [9]:
## Drop unnecessary columns
train_df = drop_unnecessary_columns(train_df)

## Define target variable and X_train and y_train
target_col = "duration_difference_min"
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

selection_df = run_feature_selection_methods(X_train, y_train)

selection_df

Feature selection methods to be used:
1. RandomForest
2. GradientBoost
3. Lasso
4. SVR
5. Ridge
--------------------------------------------------
Finished RandomForest (1/5)
Finished GradientBoost (2/5)
Finished Lasso (3/5)
Finished SVR (4/5)
Finished Ridge (5/5)


,Feature,Lasso,SVR,GradientBoost,RandomForest,Ridge,Sum
0,agency_linenum_dir_alter_encoded,1,1,1,1,1,5
1,destination_city_encoded,1,1,1,1,1,5
2,perc_within_pt_route,1,1,1,1,1,5
3,route_id,1,1,1,1,1,5
4,arrival_hour,1,1,1,1,1,5
5,Avg_Passengers_Per_Bus,1,1,1,1,1,5
6,day_encoded,1,1,1,1,1,5
7,agency_name_encoded,1,1,1,1,1,5
8,origin_city_encoded,1,1,1,1,1,5
9,origin_station_encoded,1,1,1,1,1,5


In [10]:
### Prepare validation df
val_df = pd.read_csv(r'../data/model_datasets/val_df.csv', encoding='utf-8-sig')
val_df = manipulate_df_process(val_df)
val_df = drop_unnecessary_columns(val_df)
X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col]

results_df = compare_xgb_feature_sets(
    X_train, y_train,
    X_val, y_val,
    selection_df
)

results_df

Running XGBoost on feature sets:
- All Features: 28 features
- Sum == 5: 16 features
- Sum >= 4: 27 features
--------------------------------------------------
All Features → RMSE: 13.381, MAE: 7.994, R²: 0.469
Sum == 5 → RMSE: 13.491, MAE: 8.077, R²: 0.460
Sum >= 4 → RMSE: 13.417, MAE: 8.008, R²: 0.466


,Feature_Set,Num_Features,RMSE,MAE,R2
0,All Features,28,13.380896,7.994281,0.468870
2,Sum >= 4,27,13.417070,8.008448,0.465994
1,Sum == 5,16,13.491494,8.077030,0.460053
